ESSA É O CERTO

In [ ]:
import customtkinter as ctk
import tkinter as tk
from tkinter import ttk, messagebox
import firebase_admin 
from firebase_admin import credentials, db 
from PIL import Image, ImageTk 
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import os
import subprocess
import platform
import json

# --- Configurações Iniciais ---
ctk.set_appearance_mode("light") 
ctk.set_default_color_theme("blue")

# =======================================================================
# DADOS DE EXEMPLO PARA O DASHBOARD (MATPLOTLIB)
# =======================================================================

# Dados de exemplo (podem ser substituídos por dados reais do Firebase no futuro)
estoque = {
    "Maçãs": 10,
    "Macarrão": 25,
    "Arroz": 30,
    "Feijão": 15,
    "Ervilha": 20
}

def classificar_estoque(quantidade):
    """Classifica o nível de estoque baseado na quantidade"""
    if quantidade <= 10:
        return "BAIXO", '#FF6B6B' 
    elif quantidade <= 20:
        return "MÉDIO", '#FFD166'
    else:
        return "ALTO", '#06D6A0'

# Gerar cores e status iniciais
cores = []
status_estoque = []
for produto, quantidade in estoque.items():
    status, cor = classificar_estoque(quantidade)
    cores.append(cor)
    status_estoque.append(status)

# =======================================================================
# CLASSE: TELA MOVIMENTAÇÃO DE ESTOQUE (Entrada/Saída) - CÓDIGO ATUALIZADO
# =======================================================================

class TelaMovimentacaoEstoque:
    """
    Classe responsável por gerenciar a entrada, saída e visualização 
    de estoque, abrindo como Toplevel.
    """

    def __init__(self, master_janela):
        self.master = master_janela
        
        # 1. Configuração da Janela (Substitui tk.Tk() por tk.Toplevel())
        self.janela = tk.Toplevel(master_janela)
        self.janela.title("Produtos Cadastrados")
        self.janela.geometry("750x500")
        self.janela.configure(bg="#C2C2C2")
        self.janela.grid_rowconfigure(1, weight=1)
        self.janela.grid_columnconfigure(0, weight=1)
        
        # Inicializa o Firebase e armazena referências
        self.ref_produtos, self.ref_movimentacoes = self._inicializar_firebase()
        
        if self.ref_produtos and self.ref_movimentacoes:
            self._criar_interface()
            self._carregar_produtos()
            
            # Gerenciamento de foco e visibilidade
            self.janela.protocol("WM_DELETE_WINDOW", self._fechar_e_voltar)
            self.janela.grab_set() 
            self.master.withdraw() # Esconde a janela principal
        else:
            self.janela.destroy()
            self.master.deiconify() 

    def _inicializar_firebase(self):
        """Inicializa/reusa a conexão Firebase e retorna referências."""
        try:
            if not firebase_admin._apps:
                cred = credentials.Certificate("bancochave.json")
                firebase_admin.initialize_app(cred, {
                    'databaseURL': "https://bancodedadosprojeto-b4cec-default-rtdb.firebaseio.com/"
                })
            return db.reference("produtos"), db.reference("movimentacoes")
        except Exception as e:
            messagebox.showerror("Erro ao conectar Firebase", str(e))
            return None, None

    def _criar_interface(self):
        """Cria todos os widgets da interface, adaptando as variáveis globais para self."""
        
        # --- HEADER ---
        header = tk.Frame(self.janela, bg="#236591", height=120)
        header.grid(row=0, column=0, sticky="ew")
        header.grid_propagate(False)
        header.grid_columnconfigure(0, weight=1)
        header.grid_columnconfigure(1, weight=0) # Nova coluna para o botão

        titulo = tk.Label(header, text="Estoque Pro", bg="#236591", fg="white", font=("Arial", 22, "bold"))
        titulo.grid(row=0, column=0, pady=(15, 0), sticky="n", padx=(20, 0))

        subtitulo = tk.Label(header, text="Controle total, resultado real", bg="#236591", fg="white", font=("Arial", 12))
        subtitulo.grid(row=1, column=0, pady=(0, 15), sticky="n", padx=(20, 0))
        
        # Botão de Voltar adicionado aqui
        btn_voltar = tk.Button(header, text="Voltar ao Principal", bg='#F44336', fg='white', 
                                font=("Arial", 10, "bold"), padx=15, pady=5, relief=tk.RAISED, bd=2, 
                                command=self._fechar_e_voltar)
        btn_voltar.grid(row=0, column=1, rowspan=2, sticky='n', padx=20) # Alinhado à direita
        
        # --- TABELA (TREEVIEW) ---
        colunas = ("Produto", "Quantidade")
        self.tabela = ttk.Treeview(self.janela, columns=colunas, show="headings", height=10)
        for col in colunas:
            self.tabela.heading(col, text=col)
            self.tabela.column(col, width=200, anchor="center")
            
        self.tabela.grid(row=1, column=0, sticky="nsew", padx=(10, 0), pady=10)

        scrollbar = ttk.Scrollbar(self.janela, orient="vertical", command=self.tabela.yview)
        self.tabela.configure(yscroll=scrollbar.set)
        scrollbar.grid(row=1, column=1, sticky="ns", padx=(0, 10), pady=10)

        # --- CONTROLES E BOTÕES ---
        controle_frame = tk.Frame(self.janela, bg="#C2C2C2")
        controle_frame.grid(row=2, column=0, columnspan=2, pady=10)

        tk.Label(controle_frame, text="Quantidade:", bg="#C2C2C2").grid(row=0, column=0, padx=5)
        self.qtd_input = tk.Entry(controle_frame, width=10)
        self.qtd_input.grid(row=0, column=1, padx=5)

        # Usando 'self.alterar_quantidade' em vez de 'alterar_quantidade'
        btn_remover = tk.Button(controle_frame, text="Remover", bg="#D9534F", fg="white", 
                                command=lambda: self._alterar_quantidade("remover"))
        btn_remover.grid(row=0, column=2, padx=10)

        btn_adicionar = tk.Button(controle_frame, text="Adicionar", bg="#5CB85C", fg="white", 
                                  command=lambda: self._alterar_quantidade("adicionar"))
        btn_adicionar.grid(row=0, column=3, padx=10)

        btn_atualizar = tk.Button(controle_frame, text="Atualizar Tabela", command=self._carregar_produtos)
        btn_atualizar.grid(row=0, column=4, padx=10)

        btn_historico = tk.Button(controle_frame, text="Histórico", bg="#0275D8", fg="white", command=self._abrir_historico)
        btn_historico.grid(row=0, column=5, padx=10)

    # -----------------------------------------------------------
    # MÉTODOS DE DADOS (Baseados no código fornecido)
    # -----------------------------------------------------------

    def _carregar_produtos(self):
        """Carrega produtos na tabela Treeview."""
        self.tabela.delete(*self.tabela.get_children())
        # Usando self.ref_produtos
        dados = self.ref_produtos.get()
        
        if dados:
            for id_produto, item in dados.items():
                produto_nome = item.get("nome", "-") 
                quantidade = item.get("quantidade", 0) 
                self.tabela.insert("", "end", iid=id_produto, values=(produto_nome, quantidade))
        else:
            messagebox.showinfo("Aviso", "Nenhum produto cadastrado.")

    def _registrar_historico(self, nome, tipo, qtd):
        """Registra a movimentação no nó 'movimentacoes'."""
        try:
            historico_ref = self.ref_movimentacoes  # Usando self.ref_movimentacoes
            hora_atual = datetime.now().strftime("%d/%m/%Y %H:%M:%S")
            historico_ref.push({
                "produto": nome,
                "tipo": tipo,
                "quantidade": qtd,
                "data_hora": hora_atual
            })
        except Exception as e:
            messagebox.showerror("Erro", f"Erro ao registrar histórico: {e}")

    def _alterar_quantidade(self, tipo):
        """Altera a quantidade do produto selecionado (Entrada ou Saída)."""
        selecionado = self.tabela.focus()
        if not selecionado:
            messagebox.showwarning("Aviso", "Selecione um produto.")
            return

        produto_ref = self.ref_produtos.child(selecionado) # Usando self.ref_produtos
        produto = produto_ref.get() 

        if produto is None:
            messagebox.showerror("Erro", "Produto não encontrado no banco.")
            return

        nome = produto.get("nome", "-")  
        qtd_atual = produto.get("quantidade", 0)  
        qtd_str = self.qtd_input.get() # Usando self.qtd_input

        if not qtd_str.isdigit() or int(qtd_str) <= 0:
            messagebox.showwarning("Aviso", "Digite uma quantidade válida.")
            return

        qtd = int(qtd_str)
        if tipo == "remover":
            if qtd > qtd_atual:
                messagebox.showerror("Erro", "A quantidade a remover é maior que o estoque!")
                return
            nova_qtd = qtd_atual - qtd
        else:
            nova_qtd = qtd_atual + qtd

        produto_ref.update({"quantidade": nova_qtd})

        self._registrar_historico(nome, "Entrada" if tipo == "adicionar" else "Saída", qtd) # Chamando método da classe

        self._carregar_produtos()
        self.qtd_input.delete(0, tk.END)

    def _abrir_historico(self):
        """Abre uma nova janela Toplevel para exibir o histórico de movimentações."""
        hist_janela = tk.Toplevel(self.janela)
        hist_janela.title("Histórico de Movimentações")
        hist_janela.geometry("750x400")
        hist_janela.configure(bg="#D9D9D9")
        
        colunas = ("Produto", "Tipo", "Quantidade", "Data/Hora")
        tabela_hist = ttk.Treeview(hist_janela, columns=colunas, show="headings", height=15)
        for col in colunas:
            tabela_hist.heading(col, text=col)
            tabela_hist.column(col, width=150, anchor="center")
            
        tabela_hist.pack(fill="both", expand=True, padx=(10, 0), pady=10)

        scrollbar = ttk.Scrollbar(hist_janela, orient="vertical", command=tabela_hist.yview)
        tabela_hist.configure(yscroll=scrollbar.set)
        scrollbar.pack(side="right", fill="y", padx=(0, 10), pady=10)

        try:
            dados = self.ref_movimentacoes.get() # Usando self.ref_movimentacoes
            if dados:
                # Inverte a ordem para mostrar os mais recentes primeiro
                for item in reversed(list(dados.values())):
                    tabela_hist.insert("", "end", values=(
                        item.get("produto", "-"), 
                        item.get("tipo", "-"),
                        item.get("quantidade", "-"), 
                        item.get("data_hora", "-")))
            else:
                messagebox.showinfo("Histórico", "Nenhuma movimentação registrada.")
        except Exception as e:
            messagebox.showerror("Erro", f"Erro ao carregar histórico: {e}")
            
    def _fechar_e_voltar(self):
        """Fecha a janela de movimentação e reexibe a janela principal."""
        self.janela.destroy()
        self.master.deiconify() 

# =======================================================================
# CLASSE: TELA DE MOVIMENTAÇÕES REGISTRADAS
# =======================================================================

class TelaMovimentacoes:
    def __init__(self, master_janela):
        self.master = master_janela
        self.janela = tk.Toplevel(master_janela)
        self.janela.title("Movimentações de Estoque")
        self.janela.geometry("950x700")
        self.janela.configure(bg="#C2C2C2")
        self.janela.iconbitmap("logoo-ofcc.ico")
        
        # Configurar expansão da janela
        self.janela.grid_rowconfigure(1, weight=1)
        self.janela.grid_columnconfigure(0, weight=1)
        
        # Inicializar Firebase
        self.ref_produtos, self.ref_mov = self._inicializar_firebase()
        
        if self.ref_produtos and self.ref_mov:
            self._criar_interface()
            self._carregar_dados_iniciais()
            
            # Configurar comportamento ao fechar
            self.janela.protocol("WM_DELETE_WINDOW", self._fechar_e_voltar)
            self.janela.grab_set()
            self.master.withdraw()
        else:
            self.janela.destroy()
            self.master.deiconify()
    
    def _inicializar_firebase(self):
        """Inicializa a conexão com o Firebase"""
        try:
            if not firebase_admin._apps:
                cred = credentials.Certificate("bancochave.json")
                firebase_admin.initialize_app(cred, {
                    'databaseURL': "https://bancoback-3c307-default-rtdb.firebaseio.com/"
                })
                print("✅ Firebase conectado para Movimentações!")
            
            return db.reference("produtos"), db.reference("movimentacoes")
        except Exception as e:
            messagebox.showerror("Erro Firebase", str(e))
            return None, None
    
    def _criar_interface(self):
        """Cria a interface gráfica da tela de movimentações"""
        
        # --- HEADER ---
        header = tk.Frame(self.janela, bg="#236591", height=130)
        header.grid(row=0, column=0, sticky="ew")
        header.grid_propagate(False)
        
        # Título
        tk.Label(
            header, 
            text="Estoque Pro",
            font=("Arial", 22, "bold"), 
            fg="white", 
            bg="#236591"
        ).pack(pady=(25, 0))
        
        # Subtítulo
        tk.Label(
            header, 
            text="Controle total, resultado real.",
            font=("Arial", 12), 
            fg="white", 
            bg="#236591"
        ).pack()
        
        # Botão Voltar
        btn_voltar = tk.Button(
            header,
            text="Voltar ao Principal",
            bg='#F44336',
            fg='white',
            font=("Arial", 10, "bold"),
            padx=15,
            pady=5,
            relief=tk.RAISED,
            bd=2,
            command=self._fechar_e_voltar
        )
        btn_voltar.place(relx=0.95, rely=0.5, anchor="e")
        
        # --- ÁREA PRINCIPAL ---
        self.frame_principal = tk.Frame(self.janela, bg="#ECECEC")
        self.frame_principal.grid(row=1, column=0, sticky="nsew", padx=10, pady=10)
        
        # Carregar dados iniciais
        self._carregar_produtos()
        
        # Seção de Entrada
        self._criar_secao_entrada()
        
        # Seção de Saída
        self._criar_secao_saida()
        
        # Tabela de Histórico
        self._criar_tabela_historico()
    
    def _carregar_dados_iniciais(self):
        """Carrega os dados iniciais necessários"""
        self._carregar_produtos()
        self._carregar_tabela_mov()
    
    def _carregar_produtos(self):
        """Carrega a lista de produtos do Firebase"""
        dados = self.ref_produtos.get()
        self.lista_produtos = []
        self.mapa_ids = {}
        
        if dados:
            for idp, item in dados.items():
                nome = item.get("nome", "")
                if nome:  # Só adiciona se tiver nome
                    self.lista_produtos.append(nome)
                    self.mapa_ids[nome] = idp
    
    def _criar_secao_entrada(self):
        """Cria a seção para registrar entrada de produtos"""
        # Título
        tk.Label(
            self.frame_principal, 
            text="Registrar Entrada",
            bg="#ECECEC",
            fg="#0c3564", 
            font=("Arial", 18, "bold")
        ).pack(pady=10)
        
        # Frame da entrada
        entrada_frame = tk.Frame(self.frame_principal, bg="#ECECEC")
        entrada_frame.pack(pady=5)
        
        # Produto
        tk.Label(entrada_frame, text="Produto:", bg="#ECECEC", font=("Arial", 11)).grid(row=0, column=0, sticky="w", padx=5, pady=5)
        self.cb_entrada = ttk.Combobox(entrada_frame, values=self.lista_produtos, width=45, font=("Arial", 10))
        self.cb_entrada.grid(row=0, column=1, padx=10, pady=5)
        self.cb_entrada.set("Selecione um produto...")
        
        # Quantidade
        tk.Label(entrada_frame, text="Quantidade:", bg="#ECECEC", font=("Arial", 11)).grid(row=1, column=0, sticky="w", padx=5, pady=5)
        self.entrada_qtd = tk.Entry(entrada_frame, width=25, font=("Arial", 10))
        self.entrada_qtd.grid(row=1, column=1, padx=10, pady=5)
        
        # Data
        tk.Label(entrada_frame, text="Data (DD/MM/AAAA):", bg="#ECECEC", font=("Arial", 11)).grid(row=2, column=0, sticky="w", padx=5, pady=5)
        self.entrada_data = tk.Entry(entrada_frame, width=25, font=("Arial", 10))
        self.entrada_data.grid(row=2, column=1, padx=10, pady=5)
        self.entrada_data.insert(0, self._obter_data_atual())
        
        # Nota Fiscal
        tk.Label(entrada_frame, text="Nota Fiscal:", bg="#ECECEC", font=("Arial", 11)).grid(row=3, column=0, sticky="w", padx=5, pady=5)
        self.entrada_nota = tk.Entry(entrada_frame, width=25, font=("Arial", 10))
        self.entrada_nota.grid(row=3, column=1, padx=10, pady=5)
        
        # Observação
        tk.Label(entrada_frame, text="Observação:", bg="#ECECEC", font=("Arial", 11)).grid(row=4, column=0, sticky="w", padx=5, pady=5)
        self.entrada_obs = tk.Entry(entrada_frame, width=45, font=("Arial", 10))
        self.entrada_obs.grid(row=4, column=1, padx=10, pady=5)
        
        # Botão Registrar Entrada
        btn_registrar_entrada = tk.Button(
            entrada_frame,
            text="📥 Registrar Entrada",
            command=self._registrar_entrada,
            bg="#236591",
            fg="white",
            font=("Arial", 11, "bold"),
            width=30,
            height=2,
            cursor="hand2",
            relief="flat"
        )
        btn_registrar_entrada.grid(row=5, column=0, columnspan=2, pady=15)
    
    def _criar_secao_saida(self):
        """Cria a seção para registrar saída de produtos"""
        # Título
        tk.Label(
            self.frame_principal, 
            text="Registrar Saída",
            bg="#ECECEC",
            fg="#6e0a0a", 
            font=("Arial", 18, "bold")
        ).pack(pady=(20, 10))
        
        # Frame da saída
        saida_frame = tk.Frame(self.frame_principal, bg="#ECECEC")
        saida_frame.pack(pady=5)
        
        # Produto
        tk.Label(saida_frame, text="Produto:", bg="#ECECEC", font=("Arial", 11)).grid(row=0, column=0, sticky="w", padx=5, pady=5)
        self.cb_saida = ttk.Combobox(saida_frame, values=self.lista_produtos, width=45, font=("Arial", 10))
        self.cb_saida.grid(row=0, column=1, padx=10, pady=5)
        self.cb_saida.set("Selecione um produto...")
        
        # Quantidade
        tk.Label(saida_frame, text="Quantidade:", bg="#ECECEC", font=("Arial", 11)).grid(row=1, column=0, sticky="w", padx=5, pady=5)
        self.saida_qtd = tk.Entry(saida_frame, width=25, font=("Arial", 10))
        self.saida_qtd.grid(row=1, column=1, padx=10, pady=5)
        
        # Data
        tk.Label(saida_frame, text="Data (DD/MM/AAAA):", bg="#ECECEC", font=("Arial", 11)).grid(row=2, column=0, sticky="w", padx=5, pady=5)
        self.saida_data = tk.Entry(saida_frame, width=25, font=("Arial", 10))
        self.saida_data.grid(row=2, column=1, padx=10, pady=5)
        self.saida_data.insert(0, self._obter_data_atual())
        
        # Motivo
        tk.Label(saida_frame, text="Motivo:", bg="#ECECEC", font=("Arial", 11)).grid(row=3, column=0, sticky="w", padx=5, pady=5)
        self.cb_motivo = ttk.Combobox(
            saida_frame, 
            values=["Venda", "Perda", "Uso interno", "Ajuste", "Devolução"], 
            width=35, 
            font=("Arial", 10)
        )
        self.cb_motivo.grid(row=3, column=1, padx=10, pady=5)
        self.cb_motivo.set("Selecione o motivo...")
        
        # Observação
        tk.Label(saida_frame, text="Observação:", bg="#ECECEC", font=("Arial", 11)).grid(row=4, column=0, sticky="w", padx=5, pady=5)
        self.saida_obs = tk.Entry(saida_frame, width=45, font=("Arial", 10))
        self.saida_obs.grid(row=4, column=1, padx=10, pady=5)
        
        # Botão Registrar Saída
        btn_registrar_saida = tk.Button(
            saida_frame,
            text="📤 Registrar Saída",
            command=self._registrar_saida,
            bg="#982020",
            fg="white",
            font=("Arial", 11, "bold"),
            width=30,
            height=2,
            cursor="hand2",
            relief="flat"
        )
        btn_registrar_saida.grid(row=5, column=0, columnspan=2, pady=15)
    
    def _criar_tabela_historico(self):
        """Cria a tabela de histórico de movimentações"""
        # Título
        tk.Label(
            self.frame_principal, 
            text="📋 Histórico de Movimentações",
            bg="#ECECEC", 
            fg="#17034f",
            font=("Arial", 18, "bold")
        ).pack(pady=(25, 10))
        
        # Frame da tabela com scrollbar
        tabela_frame = tk.Frame(self.frame_principal, bg="#ECECEC")
        tabela_frame.pack(fill="both", expand=True, padx=10)
        
        # Colunas
        col_mov = ("Tipo", "Produto", "Quantidade", "Data", "Motivo/Nota", "Obs")
        
        # Tabela Treeview
        self.tabela_mov = ttk.Treeview(
            tabela_frame, 
            columns=col_mov, 
            show="headings", 
            height=12,
            selectmode="browse"
        )
        
        # Configurar cabeçalhos
        for col in col_mov:
            self.tabela_mov.heading(col, text=col)
            self.tabela_mov.column(col, width=150, anchor="center")
        
        # Scrollbar vertical
        scrollbar_y = ttk.Scrollbar(tabela_frame, orient="vertical", command=self.tabela_mov.yview)
        self.tabela_mov.configure(yscrollcommand=scrollbar_y.set)
        
        # Scrollbar horizontal
        scrollbar_x = ttk.Scrollbar(tabela_frame, orient="horizontal", command=self.tabela_mov.xview)
        self.tabela_mov.configure(xscrollcommand=scrollbar_x.set)
        
        # Posicionar widgets
        self.tabela_mov.grid(row=0, column=0, sticky="nsew")
        scrollbar_y.grid(row=0, column=1, sticky="ns")
        scrollbar_x.grid(row=1, column=0, sticky="ew")
        
        # Configurar expansão
        tabela_frame.grid_rowconfigure(0, weight=1)
        tabela_frame.grid_columnconfigure(0, weight=1)
        
        # Botão Atualizar
        btn_atualizar = tk.Button(
            self.frame_principal,
            text="🔄 Atualizar Histórico",
            command=self._carregar_tabela_mov,
            bg="#0c3564",
            fg="white",
            font=("Arial", 10, "bold"),
            padx=20,
            pady=5
        )
        btn_atualizar.pack(pady=(10, 5))
    
    def _obter_data_atual(self):
        """Retorna a data atual no formato DD/MM/AAAA"""
        from datetime import datetime
        return datetime.now().strftime("%d/%m/%Y")
    
    def _registrar_entrada(self):
        """Registra uma entrada de produto no estoque"""
        prod = self.cb_entrada.get().strip()
        qtd_str = self.entrada_qtd.get().strip()
        data = self.entrada_data.get().strip()
        nota = self.entrada_nota.get().strip()
        obs = self.entrada_obs.get().strip()
        
        # Validações
        if not prod or prod == "Selecione um produto...":
            messagebox.showwarning("Atenção", "Selecione um produto válido!")
            self.cb_entrada.focus()
            return
        
        if prod not in self.mapa_ids:
            messagebox.showerror("Erro", "Produto não encontrado no banco de dados!")
            return
        
        try:
            qtd = int(qtd_str)
            if qtd <= 0:
                messagebox.showwarning("Atenção", "A quantidade deve ser maior que zero!")
                self.entrada_qtd.focus()
                return
        except ValueError:
            messagebox.showerror("Erro", "Quantidade inválida! Digite um número inteiro.")
            self.entrada_qtd.focus()
            return
        
        if not data:
            messagebox.showwarning("Atenção", "Informe a data da movimentação!")
            self.entrada_data.focus()
            return
        
        try:
            idp = self.mapa_ids[prod]
            produto_ref = self.ref_produtos.child(idp)
            produto_atual = produto_ref.get()
            
            if not produto_atual:
                messagebox.showerror("Erro", "Produto não encontrado no banco de dados!")
                return
            
            quantidade_atual = produto_atual.get("quantidade", 0)
            
            # Atualizar quantidade no estoque
            produto_ref.update({"quantidade": quantidade_atual + qtd})
            
            # Registrar movimentação
            self.ref_mov.push({
                "tipo": "Entrada",
                "produto": prod,
                "quantidade": qtd,
                "data": data,
                "nota_fiscal": nota,
                "observacao": obs,
                "estoque_anterior": quantidade_atual,
                "estoque_atual": quantidade_atual + qtd
            })
            
            messagebox.showinfo("✅ Sucesso", f"Entrada de {qtd} unidades registrada para {prod}!")
            
            # Limpar campos
            self.entrada_qtd.delete(0, tk.END)
            self.entrada_nota.delete(0, tk.END)
            self.entrada_obs.delete(0, tk.END)
            
            # Atualizar histórico
            self._carregar_tabela_mov()
            
        except Exception as e:
            messagebox.showerror("Erro", f"Falha ao registrar entrada:\n{str(e)}")
    
    def _registrar_saida(self):
        """Registra uma saída de produto do estoque"""
        prod = self.cb_saida.get().strip()
        qtd_str = self.saida_qtd.get().strip()
        data = self.saida_data.get().strip()
        motivo = self.cb_motivo.get().strip()
        obs = self.saida_obs.get().strip()
        
        # Validações
        if not prod or prod == "Selecione um produto...":
            messagebox.showwarning("Atenção", "Selecione um produto válido!")
            self.cb_saida.focus()
            return
        
        if prod not in self.mapa_ids:
            messagebox.showerror("Erro", "Produto não encontrado no banco de dados!")
            return
        
        try:
            qtd = int(qtd_str)
            if qtd <= 0:
                messagebox.showwarning("Atenção", "A quantidade deve ser maior que zero!")
                self.saida_qtd.focus()
                return
        except ValueError:
            messagebox.showerror("Erro", "Quantidade inválida! Digite um número inteiro.")
            self.saida_qtd.focus()
            return
        
        if not data:
            messagebox.showwarning("Atenção", "Informe a data da movimentação!")
            self.saida_data.focus()
            return
        
        if not motivo or motivo == "Selecione o motivo...":
            messagebox.showwarning("Atenção", "Selecione um motivo para a saída!")
            self.cb_motivo.focus()
            return
        
        try:
            idp = self.mapa_ids[prod]
            produto_ref = self.ref_produtos.child(idp)
            produto_atual = produto_ref.get()
            
            if not produto_atual:
                messagebox.showerror("Erro", "Produto não encontrado no banco de dados!")
                return
            
            quantidade_atual = produto_atual.get("quantidade", 0)
            
            # Verificar se há estoque suficiente
            if qtd > quantidade_atual:
                messagebox.showerror("Erro", f"Estoque insuficiente!\nDisponível: {quantidade_atual} unidades\nSolicitado: {qtd} unidades")
                return
            
            # Atualizar quantidade no estoque
            produto_ref.update({"quantidade": quantidade_atual - qtd})
            
            # Registrar movimentação
            self.ref_mov.push({
                "tipo": "Saída",
                "produto": prod,
                "quantidade": qtd,
                "data": data,
                "motivo": motivo,
                "observacao": obs,
                "estoque_anterior": quantidade_atual,
                "estoque_atual": quantidade_atual - qtd
            })
            
            messagebox.showinfo("✅ Sucesso", f"Saída de {qtd} unidades registrada para {prod}!\nMotivo: {motivo}")
            
            # Limpar campos
            self.saida_qtd.delete(0, tk.END)
            self.saida_obs.delete(0, tk.END)
            
            # Atualizar histórico
            self._carregar_tabela_mov()
            
        except Exception as e:
            messagebox.showerror("Erro", f"Falha ao registrar saída:\n{str(e)}")
    
    def _carregar_tabela_mov(self):
        """Carrega o histórico de movimentações na tabela"""
        try:
            # Limpar tabela atual
            for item in self.tabela_mov.get_children():
                self.tabela_mov.delete(item)
            
            # Buscar dados
            dados = self.ref_mov.get()
            
            if not dados:
                return
            
            # Adicionar dados à tabela (mais recentes primeiro)
            movimentacoes = list(dados.items())
            movimentacoes.reverse()  # Ordenar do mais recente para o mais antigo
            
            for idm, item in movimentacoes:
                motivo_ou_nota = item.get("nota_fiscal") or item.get("motivo") or "—"
                
                # Adicionar à tabela
                self.tabela_mov.insert(
                    "",
                    tk.END,
                    values=(
                        item.get("tipo", "—"),
                        item.get("produto", "—"),
                        item.get("quantidade", "—"),
                        item.get("data", "—"),
                        motivo_ou_nota,
                        item.get("observacao", "—")
                    )
                )
            
            # Atualizar contagem
            total_entradas = sum(1 for item in dados.values() if item.get("tipo") == "Entrada")
            total_saidas = sum(1 for item in dados.values() if item.get("tipo") == "Saída")
            
            # Mostrar estatísticas no título da tabela
            for widget in self.frame_principal.winfo_children():
                if isinstance(widget, tk.Label) and "Histórico de Movimentações" in widget.cget("text"):
                    widget.config(text=f"📋 Histórico de Movimentações (Entradas: {total_entradas} | Saídas: {total_saidas})")
                    break
            
        except Exception as e:
            messagebox.showerror("Erro", f"Falha ao carregar histórico:\n{str(e)}")
    
    def _fechar_e_voltar(self):
        """Fecha a janela e retorna à tela principal"""
        self.janela.destroy()
        self.master.deiconify() 
# =======================================================================
# CLASSE: TELA DE RELATÓRIOS
# =======================================================================

class TelaRelatorios:
    def __init__(self, master_janela):
        self.master = master_janela
        self.janela = tk.Toplevel(master_janela)
        self.janela.title("Relatórios - Estoque Pro")
        self.janela.geometry('650x400')
        self.janela.configure(bg="#F8F8F8")
        if os.path.exists("logoo-ofcc.ico"):
            self.janela.iconbitmap("logoo-ofcc.ico")
        
        # Configurações de expansão
        self.janela.grid_rowconfigure(1, weight=1)
        self.janela.grid_columnconfigure(0, weight=1)
        
        # Centralizar janela
        self.centralizar_janela()
        
        # Criar interface
        self.criar_interface()
        
        # Configurar comportamento ao fechar
        self.janela.protocol("WM_DELETE_WINDOW", self.fechar_e_voltar)
        self.janela.grab_set()
        self.master.withdraw()
    
    def centralizar_janela(self):
        """Centraliza a janela na tela"""
        self.janela.update_idletasks()
        largura = 650
        altura = 400
        largura_tela = self.janela.winfo_screenwidth()
        altura_tela = self.janela.winfo_screenheight()
        posx = (largura_tela // 2) - (largura // 2)
        posy = (altura_tela // 2) - (altura // 2)
        self.janela.geometry(f"{largura}x{altura}+{posx}+{posy}")
    
    def criar_interface(self):
        """Cria a interface da tela de relatórios"""
        
        # --- CABEÇALHO ---
        header = tk.Frame(self.janela, bg="#236591", height=100)
        header.grid(row=0, column=0, sticky="ew")
        header.grid_columnconfigure(0, weight=1)
        
        # Título centralizado
        titulo_frame = tk.Frame(header, bg="#236591")
        titulo_frame.place(relx=0.5, rely=0.5, anchor="center")
        
        tk.Label(
            titulo_frame,
            text="Estoque Pro",
            bg="#236591",
            fg="white",
            font=("Arial", 22, "bold")
        ).pack()
        
        tk.Label(
            titulo_frame,
            text="Controle total, resultado real",
            fg="white",
            bg="#236591",
            font=("Arial", 12)
        ).pack()
        
        # Botão Voltar
        btn_voltar = tk.Button(
            header,
            text="Voltar ao principal",
            bg='#F44336',
            fg='white',
            font=("Arial", 10, "bold"),
            padx=15,
            pady=5,
            relief=tk.RAISED,
            bd=2,
            command=self.fechar_e_voltar
        )
        btn_voltar.place(relx=0.95, rely=0.5, anchor="e")
        
        # --- ÁREA PRINCIPAL ---
        main_frame = tk.Frame(self.janela, bg="#F8F8F8")
        main_frame.grid(row=1, column=0, sticky="nsew", padx=20, pady=(10, 20))
        
        # Título do formulário
        tk.Label(
            main_frame,
            text="Visualizar Relatórios",
            bg="#F8F8F8",
            fg="#236591",
            font=("Arial", 18, "bold")
        ).pack(pady=(0, 25))
        

        # Frame para os botões
        botoes_frame = tk.Frame(main_frame, bg="#F8F8F8")
        botoes_frame.pack(pady=10)
        
        # Botão 1: Relatório Excel
        btn_excel = tk.Button(
            botoes_frame,
            text="📊 Relatório Geral em Excel",
            command=self.abrir_excel,
            bg="#28a745",
            fg="white",
            font=("Arial", 12, "bold"),
            width=30,
            height=2,
            cursor="hand2",
            relief="flat",
            activebackground="#1A4492",
            activeforeground="white"
        )
        btn_excel.grid(row=0, column=0, padx=10, pady=10)
        
        # Status frame
        status_frame = tk.Frame(main_frame, bg="#F8F8F8")
        status_frame.pack(pady=(20, 0))
        
    
    def abrir_excel(self):
        """Abre o arquivo Excel existente ou pergunta para criar novo"""
        caminho_excel = "Relatório Estoque-Pro.xlsx"
        
        # Verifica se o arquivo existe
        if os.path.exists(caminho_excel):
            try:
                sistema = platform.system()
                
                if sistema == "Windows":
                    os.startfile(caminho_excel)
                elif sistema == "Darwin":  # macOS
                    subprocess.run(["open", caminho_excel])
                else:  # Linux
                    subprocess.run(["xdg-open", caminho_excel])
                    
                messagebox.showinfo("Sucesso", "Arquivo Excel aberto com sucesso!")
                
            except Exception as e:
                messagebox.showerror("Erro", f"Não foi possível abrir o arquivo Excel.\nErro: {str(e)}")
        else:
            # Se o arquivo não existir, pergunta se quer criar um novo
            resposta = messagebox.askyesno(
                "Arquivo não encontrado", 
                f"O arquivo '{caminho_excel}' não foi encontrado.\nDeseja criar um novo arquivo Excel?"
            )
            
            if resposta:
                self.criar_novo_excel(caminho_excel)
    
    def criar_novo_excel(self, caminho):
        """Cria um novo arquivo Excel com estrutura básica"""
        try:
            import pandas as pd
            import openpyxl
            
            # Criar um DataFrame vazio com colunas básicas
            df = pd.DataFrame(columns=[
                'ID', 'Produto', 'Quantidade', 'Unidade', 
                'Fornecedor', 'Última Movimentação', 'Status'
            ])
            
            # Salvar como Excel
            df.to_excel(caminho, index=False)
            
            # Configurar formatação básica
            from openpyxl import load_workbook
            wb = load_workbook(caminho)
            ws = wb.active
            
            # Ajustar largura das colunas
            for column in ws.columns:
                max_length = 0
                column_letter = column[0].column_letter
                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                adjusted_width = min(max_length + 2, 30)
                ws.column_dimensions[column_letter].width = adjusted_width
            
            # Formatar cabeçalho
            for cell in ws[1]:
                cell.font = openpyxl.styles.Font(bold=True)
                cell.fill = openpyxl.styles.PatternFill(start_color="236591", end_color="236591", fill_type="solid")
                cell.font = openpyxl.styles.Font(color="FFFFFF", bold=True)
            
            wb.save(caminho)
            
            messagebox.showinfo("Sucesso", f"Novo arquivo Excel criado em:\n{caminho}")
            
            # Tenta abrir o arquivo recém-criado
            self.abrir_excel()
            
        except ImportError:
            messagebox.showerror("Erro", "Bibliotecas necessárias não encontradas.\nInstale: pip install pandas openpyxl")
        except Exception as e:
            messagebox.showerror("Erro", f"Não foi possível criar o arquivo Excel:\n{str(e)}")
    
    
    def fechar_e_voltar(self):
        """Fecha a janela e retorna à tela principal"""
        self.janela.destroy()
        self.master.deiconify()

# =======================================================================
# CLASSES RESTANTES
# =======================================================================

class TelaCadastroProdutos:
    def __init__(self, master_janela):
        self.master = master_janela
        self.janela = tk.Toplevel(master_janela) 
        self.janela.title("Cadastro de Produtos - Estoque Pro")
        self.configurar_janela()
        self.db_ref = self.inicializar_firebase()
        if self.db_ref:
            self.criar_interface()
            self.janela.protocol("WM_DELETE_WINDOW", self.fechar_e_voltar)
            self.janela.grab_set() 
            self.master.withdraw()
        else:
            self.janela.destroy()
            self.master.deiconify()

    def configurar_janela(self):
        largura, altura = 650, 700
        self.janela.geometry(f"{largura}x{altura}")
        self.janela.configure(bg="#C2C2C2")
        self.janela.update_idletasks()
        largura_tela, altura_tela = self.janela.winfo_screenwidth(), self.janela.winfo_screenheight()
        posx, posy = (largura_tela // 2) - (largura // 2), (altura_tela // 2) - (altura // 2)
        self.janela.geometry(f"{largura}x{altura}+{posx}+{posy}")
        self.janela.grid_rowconfigure(1, weight=1)
        self.janela.grid_columnconfigure(0, weight=1)

    def inicializar_firebase(self):
        try:
            if not firebase_admin._apps:
                cred = credentials.Certificate("bancochave.json")
                firebase_admin.initialize_app(cred, {'databaseURL': "https://bancoback-3c307-default-rtdb.firebaseio.com/"})
            return db.reference("produtos")
        except Exception as e:
            messagebox.showerror("Erro Firebase", f"Falha ao obter referência de produtos:\n{str(e)}")
            return None

    def criar_interface(self):
        header = tk.Frame(self.janela, bg="#236591", height=150); header.grid(row=0, column=0, sticky="ew"); header.grid_propagate(False)
        titulo_frame = tk.Frame(header, bg="#236591"); titulo_frame.pack(side="left", fill="y", padx=20)
        tk.Label(titulo_frame, text="Estoque Pro", bg="#236591", fg="white", font=("Arial", 22, "bold")).pack(pady=(15, 5))
        tk.Label(titulo_frame, text="Cadastro de Produtos", fg="white", bg="#236591", font=("Arial", 12)).pack(pady=(0, 15))
        
        main_frame = tk.Frame(self.janela, bg="#C2C2C2"); main_frame.grid(row=1, column=0, padx=20, pady=(10, 20))
        tk.Label(main_frame, text="Detalhes do Produto", bg="#C2C2C2", fg="#236591", font=("Arial", 18, "bold")).pack(pady=(0, 15))
        form_frame = tk.Frame(main_frame, bg="white", relief="ridge", bd=2); form_frame.pack(fill="both", expand=True, padx=10, pady=10)
        
        style = ttk.Style(); style.configure("TLabel", font=("Arial", 11), background="white")
        
        labels = ["Código do Produto:", "Nome do Produto:", "Descrição do Produto:", "Unidade (un, kg, pç):", "Quantidade:", "Quantidade Mínima:", "Fornecedor:", "Importância:"]
        self.entries = {}
        for i, texto in enumerate(labels):
            ttk.Label(form_frame, text=texto).grid(row=i, column=0, sticky="e", pady=8, padx=(20, 10))
            if texto == "Importância:":
                combo = ttk.Combobox(form_frame, values=["Baixa", "Média", "Alta"], state="readonly", width=33); combo.set("Selecione...")
                combo.grid(row=i, column=1, pady=8, padx=(0, 20)); self.entries[texto] = combo
            else:
                ent = ttk.Entry(form_frame, width=35); ent.grid(row=i, column=1, pady=8, padx=(0, 20)); self.entries[texto] = ent
        
        botao_frame = tk.Frame(form_frame, bg="white"); botao_frame.grid(row=len(labels), column=0, columnspan=2, pady=(20, 10))
        ttk.Button(botao_frame, text="Salvar", command=self.salvar_produto, width=12).grid(row=0, column=0, padx=10)
        ttk.Button(botao_frame, text="Novo", command=self.limpar_campos, width=12).grid(row=0, column=1, padx=10)
        ttk.Button(botao_frame, text="Voltar", command=self.fechar_e_voltar, width=12).grid(row=0, column=2, padx=10)
        self.janela.after(100, lambda: self.entries["Código do Produto:"].focus_set())
    
    def salvar_produto(self):
        codigo = self.entries["Código do Produto:"].get().strip(); nome = self.entries["Nome do Produto:"].get().strip()
        descricao = self.entries["Descrição do Produto:"].get().strip(); unidade = self.entries["Unidade (un, kg, pç):"].get().strip()
        quantidade = self.entries["Quantidade:"].get().strip(); quantidade_minima = self.entries["Quantidade Mínima:"].get().strip()
        fornecedor = self.entries["Fornecedor:"].get().strip(); importancia = self.entries["Importância:"].get().strip()
        if not codigo or not nome or importancia == "Selecione...": messagebox.showwarning("Aviso", "Preencha Código, Nome e Importância."); return
        try:
            qtd = int(quantidade) if quantidade else 0; qtd_min = int(quantidade_minima) if quantidade_minima else 0
            unidade_valida = unidade.lower() if unidade else 'un'
            if unidade_valida not in ['un', 'kg', 'pç', 'pc', 'lt']: unidade_valida = 'un'
            dados_produto = {"codigo": codigo, "nome": nome, "descricao": descricao, "unidade": unidade_valida, "quantidade": qtd, "quantidade_minima": qtd_min, "fornecedor": fornecedor, "importancia": importancia, "data_cadastro": db.SERVER_TIMESTAMP, "ativo": True, "ultima_atualizacao": db.SERVER_TIMESTAMP }
            
            produtos = self.db_ref.get(); codigo_existe, produto_id = False, None
            if produtos:
                for pid, pdata in produtos.items():
                    if pdata.get('codigo') == codigo:
                        codigo_existe, produto_id = True, pid; break
            
            if codigo_existe:
                self.db_ref.child(produto_id).update(dados_produto); messagebox.showinfo("Atualizado", f"Produto '{nome}' atualizado com sucesso!")
            else:
                novo = self.db_ref.push(dados_produto); messagebox.showinfo("Cadastrado", f"Produto '{nome}' salvo com o ID: {novo.key}")
            self.limpar_campos()
            
        except ValueError:
            messagebox.showerror("Erro", "Quantidade e Quantidade Mínima devem ser números inteiros válidos.")
        except Exception as e:
            messagebox.showerror("Erro Firebase", f"Falha ao salvar produto:\n{str(e)}")

    def limpar_campos(self):
        for entry in self.entries.values():
            if isinstance(entry, ttk.Entry): entry.delete(0, tk.END)
            elif isinstance(entry, ttk.Combobox): entry.set("Selecione...")
        self.entries["Código do Produto:"].focus_set()

    def fechar_e_voltar(self):
        self.janela.destroy(); self.master.deiconify()

class TelaDashboard:
    def __init__(self, master_janela, dados_usuario):
        self.master = master_janela; self.root = tk.Toplevel(master_janela); self.dados_usuario = dados_usuario
        self.usuario = dados_usuario.get("nome", "Usuário"); self.nivel_acesso = dados_usuario.get("nivel_acesso", "Funcionário")
        self.root.title("Dashboard de Estoque - Sistema de Classificação"); self.root.geometry("1400x850"); self.root.configure(bg='#f0f0f0')
        self.root.protocol("WM_DELETE_WINDOW", self.fechar_e_voltar); self.criar_frame_topo(); self.criar_dashboard(); self.root.grab_set(); self.master.withdraw()
    
    def criar_frame_topo(self):
        self.frame_topo = tk.Frame(self.root, bg='#236591', height=100); self.frame_topo.pack(fill='x'); self.frame_topo.pack_propagate(False); self.frame_topo.grid_columnconfigure(0, weight=1)
        logo_frame = tk.Frame(self.frame_topo, bg='#236591'); logo_frame.grid(row=0, column=0, sticky='w', padx=20, pady=10)
        tk.Label(logo_frame, text="Estoque Pro", font=("Arial", 24, "bold"), bg='#236591', fg='white').pack(anchor='w')
        tk.Label(logo_frame, text="Controle total. Resultado real.", font=("Arial", 11), bg='#236591', fg='#E0E0E0').pack(anchor='w')
        tk.Label(self.frame_topo, text=f"Usuário: {self.usuario} | Nível: {self.nivel_acesso}", font=("Arial", 12), bg='#236591', fg='#FFFFFF').grid(row=0, column=1, sticky='e', padx=20, pady=10)
        buttons_frame = tk.Frame(self.frame_topo, bg='#236591'); buttons_frame.grid(row=0, column=2, sticky='e', padx=20, pady=10)
        tk.Button(buttons_frame, text="Voltar ao Principal", bg='#F44336', fg='white', font=("Arial", 10, "bold"), padx=15, pady=5, relief=tk.RAISED, bd=2, command=self.fechar_e_voltar).pack(side=tk.LEFT)
        tk.Frame(self.root, bg='#1A5276', height=2).pack(fill='x')
        
    def criar_dashboard(self):
        frame_conteudo = tk.Frame(self.root, bg='#f0f0f0'); frame_conteudo.pack(fill='both', expand=True, padx=20, pady=10)
        tk.Label(frame_conteudo, text="Dashboard de Estoque com Classificação de Níveis", font=("Arial", 18, "bold"), bg='#f0f0f0', fg='#2c3e50').pack(pady=(0, 15))
        self.criar_legenda_niveis(frame_conteudo)
        frame_graficos = tk.Frame(frame_conteudo, bg='#f0f0f0'); frame_graficos.pack(expand=True, fill='both', pady=10)
        frame_pizza = tk.LabelFrame(frame_graficos, text="Distribuição do Estoque (Gráfico de Pizza com Classificação)", font=("Arial", 12, "bold"), bg='white', relief=tk.RIDGE, bd=2); frame_pizza.pack(side=tk.LEFT, expand=True, fill='both', padx=10, pady=10)
        frame_barras = tk.LabelFrame(frame_graficos, text="Quantidade por Produto (Classificação por Cores)", font=("Arial", 12, "bold"), bg='white', relief=tk.RIDGE, bd=2); frame_barras.pack(side=tk.RIGHT, expand=True, fill='both', padx=10, pady=10)
        self.criar_grafico_pizza_classificado(frame_pizza); self.criar_grafico_barras_classificado(frame_barras)
        frame_tabela = tk.LabelFrame(frame_conteudo, text="Dados do Estoque com Classificação", font=("Arial", 12, "bold"), bg='white', relief=tk.RIDGE, bd=2); frame_tabela.pack(fill='both', expand=True, pady=(10, 0))
        self.criar_tabela_classificada(frame_tabela); self.criar_rodape(frame_conteudo)

    def criar_legenda_niveis(self, parent_frame):
        frame_legenda = tk.Frame(parent_frame, bg='#f0f0f0'); frame_legenda.pack(pady=5)
        niveis = [("BAIXO (≤10 unidades)", '#FF6B6B'), ("MÉDIO (11-20 unidades)", '#FFD166'), ("ALTO (>20 unidades)", '#06D6A0')]
        for texto, cor in niveis:
            frame_nivel = tk.Frame(frame_legenda, bg='#f0f0f0'); frame_nivel.pack(side=tk.LEFT, padx=20)
            tk.Canvas(frame_nivel, width=20, height=20, bg=cor, highlightthickness=0).pack(side=tk.LEFT, padx=5)
            tk.Label(frame_nivel, text=texto, font=("Arial", 9), bg='#f0f0f0', fg='#34495e').pack(side=tk.LEFT)

    def criar_grafico_pizza_classificado(self, frame):
        produtos = list(estoque.keys()); quantidades = list(estoque.values())
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), dpi=100)
        ax1.pie(quantidades, labels=produtos, colors=cores, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 8})
        ax1.set_title('Distribuição por Produto', fontsize=14, fontweight='bold', pad=20); ax1.axis('equal')
        contador_niveis = {"BAIXO": 0, "MÉDIO": 0, "ALTO": 0}
        for status in status_estoque: contador_niveis[status] += 1
        niveis = ["BAIXO", "MÉDIO", "ALTO"]; cores_niveis = ['#FF6B6B', '#FFD166', '#06D6A0']
        quantidades_niveis = [contador_niveis[n] for n in niveis]
        dados_filtrados = [(n, q, c) for n, q, c in zip(niveis, quantidades_niveis, cores_niveis) if q > 0]
        if dados_filtrados:
            niveis_f, quantidades_f, cores_f = zip(*dados_filtrados)
            wedges2, texts2, autotexts2 = ax2.pie(quantidades_f, labels=niveis_f, colors=cores_f, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 9, 'fontweight': 'bold'})
            ax2.set_title('Distribuição por Nível de Estoque', fontsize=14, fontweight='bold', pad=20); plt.setp(autotexts2, size=9, weight="bold", color='white'); ax2.axis('equal')
            ax2.legend(wedges2, [f'{n}: {q} produto(s)' for n, q in zip(niveis_f, quantidades_f)], title="Níveis", loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))
        plt.tight_layout()
        canvas = FigureCanvasTkAgg(fig, frame); canvas.draw(); canvas.get_tk_widget().pack(expand=True, fill='both', padx=10, pady=10)

    def criar_grafico_barras_classificado(self, frame):
        produtos = list(estoque.keys()); quantidades = list(estoque.values())
        fig, ax = plt.subplots(figsize=(7, 5), dpi=100)
        bars = ax.bar(produtos, quantidades, color=cores, edgecolor='black', linewidth=1.5)
        for i, (bar, produto, quantidade, status) in enumerate(zip(bars, produtos, quantidades, status_estoque)):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, height + 0.5, f'{quantidade} ({status})', ha='center', va='bottom', fontsize=10, fontweight='bold')
        ax.set_title('Quantidade por Produto (Classificação por Cores)', fontsize=14, fontweight='bold', pad=20)
        ax.set_xlabel('Produtos', fontsize=12, fontweight='bold'); ax.set_ylabel('Quantidade (pacotes)', fontsize=12, fontweight='bold')
        ax.grid(True, axis='y', linestyle='--', alpha=0.7)
        ax.axhline(y=10, color='red', linestyle='--', alpha=0.5, linewidth=1, label='Limite BAIXO (10)')
        ax.axhline(y=20, color='orange', linestyle='--', alpha=0.5, linewidth=1, label='Limite MÉDIO (20)')
        ax.legend(); ax.set_ylim(0, max(quantidades) + 5)
        plt.tight_layout()
        canvas = FigureCanvasTkAgg(fig, frame); canvas.draw(); canvas.get_tk_widget().pack(expand=True, fill='both', padx=10, pady=10)

    def criar_tabela_classificada(self, frame):
        colunas = ('Produto', 'Quantidade', 'Nível Estoque', 'Status'); tree = ttk.Treeview(frame, columns=colunas, show='headings', height=6)
        tree.heading('Produto', text='Produto'); tree.heading('Quantidade', text='Quantidade (pacotes)'); tree.heading('Nível Estoque', text='Nível Estoque'); tree.heading('Status', text='Status')
        tree.column('Produto', width=150, anchor='center'); tree.column('Quantidade', width=120, anchor='center'); tree.column('Nível Estoque', width=120, anchor='center'); tree.column('Status', width=100, anchor='center')
        for produto, quantidade in estoque.items():
            nivel, cor = classificar_estoque(quantidade); emoji = "🔴" if nivel == "BAIXO" else ("🟡" if nivel == "MÉDIO" else "🟢")
            tag = 'baixo' if nivel == "BAIXO" else ('medio' if nivel == "MÉDIO" else 'alto')
            tree.insert('', 'end', values=(produto, quantidade, nivel, emoji), tags=(tag,))
        tree.tag_configure('baixo', background='#ffebee'); tree.tag_configure('medio', background='#fff3e0'); tree.tag_configure('alto', background='#e8f5e9')
        scrollbar = ttk.Scrollbar(frame, orient="vertical", command=tree.yview); tree.configure(yscrollcommand=scrollbar.set)
        tree.pack(side='left', fill='both', expand=True, padx=10, pady=10); scrollbar.pack(side='right', fill='y')

    def criar_rodape(self, parent_frame):
        total_itens = sum(estoque.values()); total_produtos = len(estoque)
        contador_niveis = {"BAIXO": 0, "MÉDIO": 0, "ALTO": 0}
        for quantidade in estoque.values(): nivel, _ = classificar_estoque(quantidade); contador_niveis[nivel] += 1
        frame_estatisticas = tk.Frame(parent_frame, bg='#ecf0f1', relief=tk.RIDGE, bd=1); frame_estatisticas.pack(fill='x', pady=(10, 0))
        estatisticas_texto = f"Total de itens: {total_itens} pacotes | Total de produtos: {total_produtos} | Usuário: {self.usuario} ({self.nivel_acesso})"
        tk.Label(frame_estatisticas, text=estatisticas_texto, font=("Arial", 11, "bold"), bg='#ecf0f1', fg='#2c3e50').pack(pady=5)
        frame_niveis = tk.Frame(frame_estatisticas, bg='#ecf0f1'); frame_niveis.pack(pady=5)
        for nivel, cor in [("BAIXO", '#FF6B6B'), ("MÉDIO", '#FFD166'), ("ALTO", '#06D6A0')]:
            count = contador_niveis[nivel]; frame_nivel = tk.Frame(frame_niveis, bg='#ecf0f1'); frame_nivel.pack(side=tk.LEFT, padx=15)
            tk.Label(frame_nivel, text=f"{nivel}: {count} produto(s)", font=("Arial", 10, "bold"), bg='#ecf0f1', fg=cor).pack()
        produtos_baixo = [produto for produto, quantidade in estoque.items() if classificar_estoque(quantidade)[0] == "BAIXO"]
        if produtos_baixo:
            frame_alerta = tk.Frame(parent_frame, bg='#ffebee', relief=tk.RIDGE, bd=1); frame_alerta.pack(fill='x', pady=5)
            alerta_texto = f"⚠️ ALERTA: Estoque BAIXO nos produtos: {', '.join(produtos_baixo)}"
            tk.Label(frame_alerta, text=alerta_texto, font=("Arial", 10, "bold"), bg='#ffebee', fg='#d32f2f').pack(pady=5)
 
    def fechar_e_voltar(self):
        self.root.destroy(); self.master.deiconify() 

# =======================================================================
# CLASSE: TELA PRINCIPAL ADMIN (ATUALIZADA COM TODOS OS BOTÕES)
# =======================================================================

# =======================================================================
# CLASSE: TELA PRINCIPAL ADMIN
# =======================================================================

class TelaPrincipalAdmin:
    def __init__(self, dados_usuario):
        self.janela = ctk.CTk()
        self.dados_usuario = dados_usuario
        self.janela.title("Estoque Pro - Administrador")
        self.janela.geometry('950x550')
        self.janela.configure(fg_color="#F8F8F8")
        if os.path.exists("logoo-ofcc.ico"):
            self.janela.iconbitmap("logoo-ofcc.ico")
        self.janela.grid_rowconfigure(1, weight=1)
        self.janela.grid_columnconfigure(0, weight=1)
        self.img_tk = None
        self.criar_interface_principal()
        
    def criar_interface_principal(self):
        ALTURA_MAX_IMAGEM = 60
        try:
            img_pil = Image.open("logoo-ofcc.png")
            ratio = ALTURA_MAX_IMAGEM / img_pil.height
            nova_largura = int(img_pil.width * ratio)
            img_pil = img_pil.resize((nova_largura, ALTURA_MAX_IMAGEM), Image.Resampling.LANCZOS)
            self.img_tk = ImageTk.PhotoImage(img_pil)
        except Exception:
            self.img_tk = None

        # Header
        header = tk.Frame(self.janela, bg="#236591", height=100)
        header.grid(row=0, column=0, sticky="ew")
        header.grid_columnconfigure(0, weight=1)
        header.grid_columnconfigure(1, weight=1)
        
        if self.img_tk:
            tk.Label(header, image=self.img_tk, bg="#236591").grid(row=0, column=0, rowspan=2, padx=(20, 10), pady=10, sticky="w")

        titulo_frame = tk.Frame(header, bg="#236591")
        titulo_frame.grid(row=0, column=1 if self.img_tk else 0, rowspan=2, sticky="w", padx=20)
        
        tk.Label(titulo_frame, text="Estoque Pro", bg="#236591", fg="white", font=("Arial", 20, "bold")).pack(pady=(5, 0))
        tk.Label(titulo_frame, text=f"Usuário: {self.dados_usuario.get('nome')} | Nível: {self.dados_usuario.get('nivel_acesso')}", 
                fg="white", bg="#236591", font=("Arial", 12)).pack(pady=(0, 5))

        # Área principal
        main_frame = tk.Frame(self.janela, bg="#F8F8F8")
        main_frame.grid(row=1, column=0, sticky="nsew", padx=20, pady=(10, 20))
        
        tk.Label(main_frame, text="Tela de Acesso - ADMINISTRADOR", bg="#F8F8F8", fg="#236591", font=("Arial", 18, "bold")).pack(pady=(0, 15))

        # Frame dos botões
        botoes_frame = tk.Frame(main_frame, bg="#F8F8F8")
        botoes_frame.pack(pady=10)
        
        for i in range(3):
            botoes_frame.grid_columnconfigure(i, weight=1)
        
        # Botões com funcionalidades
        btn1 = tk.Button(botoes_frame, text="Cadastro de Produtos", width=20, bg="#236591", height=5, 
                        fg="white", font=("Arial", 15), command=self.abrir_cadastro_produtos)
        btn1.grid(row=0, column=0, padx=5, pady=5)
        
        btn2 = tk.Button(botoes_frame, text="Entrada e Saída", width=20, bg="#236591", height=5, 
                        fg="white", font=("Arial", 15), command=self.abrir_movimentacao_estoque)
        btn2.grid(row=0, column=1, padx=5, pady=5)
        
        # BOTÃO ATUALIZADO: MOVIMENTAÇÕES REGISTRADAS
        btn3 = tk.Button(botoes_frame, text="Movimentações Registradas", width=25, bg="#236591", height=5, 
                        fg="white", font=("Arial", 15), command=self.abrir_movimentacoes_registradas)
        btn3.grid(row=0, column=2, padx=5, pady=5)
        
        btn4 = tk.Button(botoes_frame, text="Dashboard", width=20, bg="#236591", height=5, 
                        fg="white", font=("Arial", 15), command=self.abrir_dashboard)
        btn4.grid(row=1, column=0, padx=5, pady=5)
        
        # BOTÃO ATUALIZADO: RELATÓRIOS
        btn5 = tk.Button(botoes_frame, text="Relatórios", width=20, bg="#236591", height=5, 
                        fg="white", font=("Arial", 15), command=self.abrir_relatorios)
        btn5.grid(row=1, column=1, padx=5, pady=5)
        
        btn_logoff = tk.Button(botoes_frame, text="Logoff", width=25, bg="#F44336", height=5, 
                                fg="white", font=("Arial", 15), command=self.fazer_logoff)
        btn_logoff.grid(row=1, column=2, padx=5, pady=5)
    
    def fazer_logoff(self):
        self.janela.destroy()
        LoginApp()
    
    def abrir_dashboard(self):
        self.janela.withdraw()
        try:
            TelaDashboard(self.janela, self.dados_usuario)
        except Exception as e:
            messagebox.showerror("Erro", f"Não foi possível abrir o Dashboard: {e}")
            self.janela.deiconify()
    
    def abrir_cadastro_produtos(self):
        self.janela.withdraw()
        try:
            TelaCadastroProdutos(self.janela)
        except Exception as e:
            messagebox.showerror("Erro", f"Não foi possível abrir o Cadastro de Produtos: {e}")
            self.janela.deiconify()
    
    def abrir_movimentacao_estoque(self):
        self.janela.withdraw()
        try:
            TelaMovimentacaoEstoque(self.janela)
        except Exception as e:
            messagebox.showerror("Erro", f"Não foi possível abrir o Controle de Estoque: {e}")
            self.janela.deiconify()
    
    # NOVO MÉTODO: MOVIMENTAÇÕES REGISTRADAS
    def abrir_movimentacoes_registradas(self):
        self.janela.withdraw()
        try:
            TelaMovimentacoes(self.janela)
        except Exception as e:
            messagebox.showerror("Erro", f"Não foi possível abrir as Movimentações: {e}")
            self.janela.deiconify()
    
    # NOVO MÉTODO: RELATÓRIOS
    def abrir_relatorios(self):
        self.janela.withdraw()
        try:
            TelaRelatorios(self.janela)
        except Exception as e:
            messagebox.showerror("Erro", f"Não foi possível abrir os Relatórios: {e}")
            self.janela.deiconify()
    
    def iniciar(self):
        self.janela.mainloop()

# =======================================================================
# CLASSE: TELA PRINCIPAL FUNCIONÁRIO
# =======================================================================

class TelaSistemaFuncionario:
    def __init__(self, dados_usuario):
        # Usando CTk() para consistência com o restante do código
        self.janela = ctk.CTk() 
        self.dados_usuario = dados_usuario
        self.janela.title("Estoque Pro - Funcionário")
        self.janela.geometry('1350x550') 
        self.janela.configure(fg_color="#F8F8F8")
        
        if os.path.exists("logoo-ofcc.ico"):
            self.janela.iconbitmap("logoo-ofcc.ico")
            
        self.janela.grid_rowconfigure(1, weight=1)
        self.janela.grid_columnconfigure(0, weight=1)
        self.img_tk = None
        self.criar_interface_principal()
        
    def criar_interface_principal(self):
        ALTURA_MAX_IMAGEM = 60
        try:
            img_pil = Image.open("logoo-ofcc.png")
            ratio = ALTURA_MAX_IMAGEM / img_pil.height
            nova_largura = int(img_pil.width * ratio)
            img_pil = img_pil.resize((nova_largura, ALTURA_MAX_IMAGEM), Image.Resampling.LANCZOS)
            self.img_tk = ImageTk.PhotoImage(img_pil)
        except Exception:
            self.img_tk = None

        # ================= HEADER ====================
        header = tk.Frame(self.janela, bg="#236591", height=100)
        header.grid(row=0, column=0, sticky="ew")
        header.grid_columnconfigure(0, weight=1)
        header.grid_columnconfigure(1, weight=1)
        
        if self.img_tk:
            tk.Label(header, image=self.img_tk, bg="#236591").grid(row=0, column=0, rowspan=2, padx=(20, 10), pady=10, sticky="w")

        titulo_frame = tk.Frame(header, bg="#236591")
        titulo_frame.grid(row=0, column=1 if self.img_tk else 0, rowspan=2, sticky="w", padx=20) 
        
        tk.Label(titulo_frame, text="Estoque Pro", bg="#236591", fg="white", font=("Arial", 20, "bold")).pack(pady=(5, 0))
        tk.Label(
            titulo_frame, 
            text=f"Usuário: {self.dados_usuario.get('nome')} | Nível: {self.dados_usuario.get('nivel_acesso', 'Funcionário')}", 
            fg="white", 
            bg="#236591", 
            font=("Arial", 12)
        ).pack(pady=(0, 5))

        # ================= MAIN ======================
        main_frame = tk.Frame(self.janela, bg="#F8F8F8")
        main_frame.grid(row=1, column=0, sticky="nsew", padx=20, pady=(10, 20))
        
        tk.Label(
            main_frame, 
            text="Tela de Acesso - FUNCIONÁRIO", 
            bg="#F8F8F8", 
            fg="#236591", 
            font=("Arial", 18, "bold")
        ).pack(pady=(0, 15))

        # Frame dos botões
        botoes_frame = tk.Frame(main_frame, bg="#F8F8F8")
        botoes_frame.pack(pady=30) 

        # *** CONFIGURAÇÃO DO GRID PARA 6 BOTÕES (0 a 5) ***
        for i in range(6): 
            botoes_frame.grid_columnconfigure(i, weight=1)

        # ------------------ BOTÕES ------------------
        
        # 1. Entradas e Saídas (COLUNA 0)
        btn_mov_estoque = tk.Button(
            botoes_frame, 
            text="Entrada e Saída", 
            width=20, bg="#236591", height=5, 
            fg="white", font=("Arial", 15), 
            command=self.abrir_movimentacao_estoque
        )
        btn_mov_estoque.grid(row=0, column=0, padx=10, pady=10)

        # 2. Movimentações Registradas (COLUNA 1)
        btn_mov_registradas = tk.Button(
            botoes_frame, 
            text="Movimentações Registradas", 
            width=25, bg="#236591", height=5, 
            fg="white", font=("Arial", 15),
            command=self.abrir_movimentacoes_registradas
        )
        btn_mov_registradas.grid(row=0, column=1, padx=10, pady=10)
        
        # 3. Dashboard (Visualização) (COLUNA 2)
        btn_dashboard = tk.Button(
            botoes_frame, 
            text="Dashboard", 
            width=20, bg="#236591", height=5, 
            fg="white", font=("Arial", 15),
            command=self.abrir_dashboard
        )
        btn_dashboard.grid(row=0, column=2, padx=10, pady=10)
        
        # 4. Relatórios (Primeiro Botão) (COLUNA 3)
        btn_relatorios_1 = tk.Button(
            botoes_frame, 
            text="Relatórios", 
            width=25, bg="#236591", height=5, 
            fg="white", font=("Arial", 15),
            command=self.abrir_relatorios_funcionario 
        )
        btn_relatorios_1.grid(row=0, column=3, padx=10, pady=10)
        
        # Botões Adicionais (apenas para preencher o layout de 6 colunas, se necessário)
        # Exemplo de botão 'Vazio' ou 'Consultar Estoque' para Coluna 4
        btn_vazio = tk.Button(
            botoes_frame, 
            text="Consultar Estoque", 
            width=20, bg="#6495ED", height=5, 
            fg="white", font=("Arial", 15),
            command=lambda: messagebox.showinfo("Consulta", "Função Consultar Estoque - Mock")
        )
        btn_vazio.grid(row=0, column=4, padx=10, pady=10)
        
        # 6. Logoff (ÚLTIMA COLUNA - COLUNA 5)
        btn_logoff = tk.Button(
            botoes_frame, 
            text="Logoff", 
            width=20, bg="#F44336", height=5, 
            fg="white", font=("Arial", 15),
            command=self.fazer_logoff
        )
        btn_logoff.grid(row=0, column=5, padx=10, pady=10) 

    
    def fazer_logoff(self):
        """Fecha a tela atual e retorna para a tela de login."""
        self.janela.destroy()
        LoginApp() 

    def abrir_dashboard(self):
        """Abre a tela de Dashboard (apenas visualização de dados)."""
        self.janela.withdraw()
        try:
            TelaDashboard(self.janela, self.dados_usuario) 
        except Exception as e:
            messagebox.showerror("Erro", f"Não foi possível abrir o Dashboard: {e}")
            self.janela.deiconify()

    def abrir_movimentacao_estoque(self):
        """Abre a tela de Entradas e Saídas."""
        self.janela.withdraw()
        try:
            TelaMovimentacaoEstoque(self.janela) 
        except Exception as e:
            messagebox.showerror("Erro", f"Não foi possível abrir o Controle de Estoque: {e}")
            self.janela.deiconify()
    
    def abrir_movimentacoes_registradas(self):
        """Abre o histórico de movimentações registradas."""
        self.janela.withdraw()
        try:
            TelaMovimentacoes(self.janela) 
        except Exception as e:
            messagebox.showerror("Erro", f"Não foi possível abrir as Movimentações: {e}")
            self.janela.deiconify()
    
    def abrir_relatorios_funcionario(self):
        """Abre a tela de relatórios restrita para Funcionários."""
        self.janela.withdraw()
        try:
            TelaRelatorios(self.janela)
        except Exception as e:
            messagebox.showerror("Erro", f"Não foi possível abrir os Relatórios: {e}")
            self.janela.deiconify()
            
    
    def iniciar(self):
        """Inicia o loop principal da interface."""
        self.janela.mainloop()

# =======================================================================
# CLASSE: TELA DE CADASTRO
# (Sem alterações)
# =======================================================================
class TelaCadastro:
    def __init__(self, master_janela):
        self.master = master_janela
        self.janela = tk.Toplevel(master_janela)
        self.janela.title("Cadastro de Usuários")
        if os.path.exists("logoo-ofcc.ico"):
            self.janela.iconbitmap("logoo-ofcc.ico")
        
        largura, altura = 600, 350
        self.janela.geometry(f"{largura}x{altura}")
        self.janela.configure(bg="#C2C2C2")
        
        # Centralizar
        self.janela.update_idletasks()
        largura_tela, altura_tela = self.janela.winfo_screenwidth(), self.janela.winfo_screenheight()
        posx, posy = (largura_tela // 2) - (largura // 2), (altura_tela // 2) - (altura // 2)
        self.janela.geometry(f"{largura}x{altura}+{posx}+{posy}")
        
        self.janela.grid_rowconfigure(1, weight=1)
        self.janela.grid_columnconfigure(0, weight=1)
        
        self.firebase_ref = self._inicializar_firebase()
        self.entradas = {}
        self.criar_interface()
        self.janela.grab_set()
        self.master.withdraw()

    def _inicializar_firebase(self):
        try:
            if not firebase_admin._apps:
                cred = credentials.Certificate("bancochave.json")
                firebase_admin.initialize_app(cred, {'databaseURL': "https://bancoback-3c307-default-rtdb.firebaseio.com/"})
            return db.reference("usuarios")
        except FileNotFoundError:
            messagebox.showerror("Erro", "Arquivo 'bancochave.json' não encontrado!")
            return None
        except Exception as e:
            messagebox.showerror("Erro Firebase", f"Falha na conexão:\n{str(e)}")
            return None

    def criar_interface(self):
        header = tk.Frame(self.janela, bg="#236591", height=100)
        header.grid(row=0, column=0, sticky="ew")
        header.grid_propagate(False)
        header.grid_columnconfigure(0, weight=1)
        header.grid_columnconfigure(1, weight=1)
        
        titulo_frame = tk.Frame(header, bg="#236591")
        titulo_frame.place(relx=0.5, rely=0.5, anchor="center")
        
        tk.Label(titulo_frame, text="Estoque Pro", bg="#236591", fg="white", font=("Arial", 20, "bold")).pack()
        tk.Label(titulo_frame, text="Cadastro de Usuários", fg="white", bg="#236591", font=("Arial", 12)).pack()
        
        # Estilo
        style = ttk.Style()
        style.configure("TLabel", font=("Arial", 13), background="#C2C2C2")
        style.configure("TEntry", padding=3)
        style.configure("TButton", font=("Arial", 10, "bold"), padding=6)
        
        frame_form = ttk.Frame(self.janela, padding=20)
        frame_form.grid(row=1, column=0)
        
        tk.Label(frame_form, text="Detalhes do Novo Usuário", font=("Arial", 18, "bold")).grid(row=0, column=0, columnspan=2, pady=(0, 20))

        labels = ["Nome:", "Usuário:", "Senha:", "Nível de Acesso:"]
        for i, texto in enumerate(labels):
            ttk.Label(frame_form, text=texto).grid(row=i+1, column=0, sticky="e", pady=5, padx=10)
            
            if texto == "Nível de Acesso:":
                ent = ttk.Combobox(frame_form, values=["Administrador", "Funcionário"], state="readonly", width=37)
                ent.set("Funcionário")
            elif texto == "Senha:":
                ent = ttk.Entry(frame_form, show="*", width=40)
            else:
                ent = ttk.Entry(frame_form, width=40)
                
            ent.grid(row=i+1, column=1, pady=5, padx=10)
            self.entradas[texto[:-1].lower()] = ent
        
        # Botões
        botao_frame = ttk.Frame(frame_form, padding=10)
        botao_frame.grid(row=len(labels)+1, column=0, columnspan=2, pady=(10, 0))
        
        ttk.Button(botao_frame, text="Salvar", command=self.salvar_usuario).grid(row=0, column=0, padx=10)
        ttk.Button(botao_frame, text="Novo", command=self.limpar_campos).grid(row=0, column=1, padx=10)
        ttk.Button(botao_frame, text="Voltar ao Login", command=self.fechar_e_voltar).grid(row=0, column=2, padx=10)
        
        self.janela.after(100, lambda: self.entradas["nome"].focus())

    def salvar_usuario(self):
        if self.firebase_ref is None:
            messagebox.showerror("Erro", "Conexão com Firebase indisponível.")
            return
        
        dados = {
            "nome": self.entradas["nome"].get().strip(),
            "usuario": self.entradas["usuário"].get().strip(),
            "senha": self.entradas["senha"].get(),
            "nivel_acesso": self.entradas["nível de acesso"].get(),
            "data_cadastro": datetime.now().strftime("%d/%m/%Y %H:%M")
        }
        
        if not all(dados.values()) or dados["nivel_acesso"] == "Selecione o nível...":
            messagebox.showwarning("Aviso", "Preencha todos os campos corretamente.")
            return
        
        try:
            usuarios_existentes = self.firebase_ref.get()
            if usuarios_existentes:
                for user_id, user_data in usuarios_existentes.items():
                    if user_data.get('usuario') == dados['usuario']:
                        messagebox.showerror("Erro de Cadastro", f"O usuário '{dados['usuario']}' já existe.")
                        return
            
            self.firebase_ref.push(dados)
            messagebox.showinfo("Sucesso", "Usuário cadastrado no Firebase!")
            self.limpar_campos()
            
        except Exception as e:
            messagebox.showerror("Erro Firebase", f"Falha ao salvar usuário:\n{str(e)}")

    def limpar_campos(self):
        for ent in self.entradas.values():
            if isinstance(ent, ttk.Entry):
                ent.delete(0, tk.END)
            elif isinstance(ent, ttk.Combobox):
                ent.set("Funcionário")
        self.entradas["nome"].focus()
    
    def fechar_e_voltar(self):
        self.janela.destroy()
        self.master.deiconify()

# =======================================================================
# CLASSE: TELA DE LOGIN (Sem grandes alterações, mas crucial na lógica)
# =======================================================================
class LoginApp:
    def __init__(self):
        self.janela = ctk.CTk()
        self.janela.title("Tela de Login - Estoque Pro")
        largura, altura = 600, 550
        self.janela.geometry(f"{largura}x{altura}")
        self.janela.configure(bg="#C2C2C2")
        
        # Centralizar
        self.janela.update_idletasks()
        largura_tela, altura_tela = self.janela.winfo_screenwidth(), self.janela.winfo_screenheight()
        posx, posy = (largura_tela // 2) - (largura // 2), (altura_tela // 2) - (altura // 2)
        self.janela.geometry(f"{largura}x{altura}+{posx}+{posy}")
        
        self.janela.grid_rowconfigure(1, weight=1)
        self.janela.grid_columnconfigure(0, weight=1)
        self.janela.configure(fg_color="#F0F0F0")
        
        if os.path.exists("logoo-ofcc.ico"):
            self.janela.iconbitmap("logoo-ofcc.ico")
        
        self.img_tk = None
        self.db = None
        self.firebase_ok = self.inicializar_firebase()
        self.criar_interface()
        self.janela.mainloop() # INICIA O LOOP AQUI
    
    def inicializar_firebase(self):
        try:
            if not firebase_admin._apps:
                cred = credentials.Certificate("bancochave.json") 
                firebase_admin.initialize_app(cred, {
                    'databaseURL': "https://bancoback-3c307-default-rtdb.firebaseio.com/"
                })
            self.db = db.reference()
            print("✅ Firebase Realtime Database conectado!")
            return True
        except FileNotFoundError:
            messagebox.showerror("Erro", "Arquivo 'bancochave.json' não encontrado!")
            return False
        except Exception as e:
            messagebox.showerror("Erro Firebase", f"Falha na conexão:\n{str(e)}")
            return False
    
    def criar_interface(self):
        ALTURA_MAX_IMAGEM = 60
        try:
            img_pil = Image.open("logoo-ofcc.png") 
            ratio = ALTURA_MAX_IMAGEM / img_pil.height
            nova_largura = int(img_pil.width * ratio)
            img_pil = img_pil.resize((nova_largura, ALTURA_MAX_IMAGEM), Image.Resampling.LANCZOS)
            self.img_tk = ImageTk.PhotoImage(img_pil)
        except Exception:
            self.img_tk = None

        # Frame superior
        frame_topo = ctk.CTkFrame(self.janela, fg_color="#236591", height=80, corner_radius=0)
        frame_topo.pack(fill="x", pady=0)
        frame_topo.grid_columnconfigure(0, weight=0)
        frame_topo.grid_columnconfigure(1, weight=1)
        
        if self.img_tk:
            ctk.CTkLabel(frame_topo, image=self.img_tk, text="", fg_color="transparent").grid(
                row=0, column=0, rowspan=2, padx=(20, 10), pady=10, sticky="w")
        
        ctk.CTkLabel(frame_topo, text="Estoque Pro", fg_color="transparent", text_color="white", 
                      font=("Arial", 20, "bold")).grid(row=0, column=1, pady=(15, 5), sticky="w")
        ctk.CTkLabel(frame_topo, text="Controle total, resultado real", fg_color="transparent", 
                      text_color="white", font=("Arial", 12)).grid(row=1, column=1, pady=(0, 15), sticky="w")
        
        # Conteúdo principal
        frame_conteudo = ctk.CTkFrame(self.janela, fg_color="#F0F0F0")
        frame_conteudo.pack(fill="both", expand=True, padx=20, pady=15)
        
        frame_azul_conteudo = ctk.CTkFrame(frame_conteudo, fg_color="#C2C2C2", corner_radius=10)
        frame_azul_conteudo.pack(fill="both", expand=True)
        frame_azul_conteudo.grid_columnconfigure(1, weight=1)
        frame_azul_conteudo.grid_rowconfigure(1, weight=1)
        
        # Formulário de login
        frame_form = ctk.CTkFrame(frame_azul_conteudo, fg_color="#236591", corner_radius=10)
        frame_form.grid(row=1, column=1, sticky="n", padx=20, pady=20)
        
        for i in range(7):
            frame_form.grid_rowconfigure(i, weight=1)
        for i in range(3):
            frame_form.grid_columnconfigure(i, weight=1)
        
        ctk.CTkLabel(frame_form, text="Faça seu Login", fg_color="transparent", 
                      text_color="white", font=("Arial", 16, "bold")).grid(row=0, column=0, columnspan=3, pady=(20, 10))
        
        # Usuário
        ctk.CTkLabel(frame_form, text="Usuário:", fg_color="transparent", 
                      text_color="white", font=("Arial", 12)).grid(row=1, column=0, padx=(20, 10), pady=10, sticky="e")
        
        self.entrada_usuario = ctk.CTkEntry(frame_form, width=200, height=35, fg_color="white", 
                                             text_color="#333", border_width=1, font=("Arial", 12))
        self.entrada_usuario.grid(row=1, column=1, columnspan=2, padx=(0, 20), pady=10, sticky="w")
        
        # Senha
        ctk.CTkLabel(frame_form, text="Senha:", fg_color="transparent", 
                      text_color="white", font=("Arial", 12)).grid(row=2, column=0, padx=(20, 10), pady=10, sticky="e")
        
        self.entrada_senha = ctk.CTkEntry(frame_form, width=200, height=35, show="•", fg_color="white", 
                                          text_color="#333", border_width=1, font=("Arial", 12))
        self.entrada_senha.grid(row=2, column=1, columnspan=2, padx=(0, 20), pady=10, sticky="w")
        
        # Cargo
        ctk.CTkLabel(frame_form, text="Cargo:", fg_color="transparent", 
                      text_color="white", font=("Arial", 12)).grid(row=3, column=0, padx=(20, 10), pady=10, sticky="e")
        
        frame_radio = ctk.CTkFrame(frame_form, fg_color="transparent")
        frame_radio.grid(row=3, column=1, columnspan=2, padx=(0, 20), pady=10, sticky="w")
        
        self.cargo_selecionado = tk.StringVar(value="Funcionário")
        
        ctk.CTkRadioButton(frame_radio, text="Funcionário", variable=self.cargo_selecionado, 
                             value="Funcionário", fg_color="white", border_color="white", 
                             hover_color="#4A8ABF", text_color="white", font=("Arial", 11)).pack(side="left", padx=(0, 15))
        
        ctk.CTkRadioButton(frame_radio, text="Administrador", variable=self.cargo_selecionado, 
                             value="Administrador", fg_color="white", border_color="white", 
                             hover_color="#4A8ABF", text_color="white", font=("Arial", 11)).pack(side="left")
        
        # Botão Login
        btn_login = ctk.CTkButton(frame_form, text="ENTRAR", font=("Arial", 14, "bold"), 
                                     fg_color="white", text_color="#236591", hover_color="#E0E0EE", 
                                     height=40, width=150, command=self.fazer_login)
        btn_login.grid(row=4, column=0, columnspan=3, pady=(20, 5))
        
        # Botão Cadastro
        btn_cadastro = ctk.CTkButton(frame_form, text="Cadastre-se", font=("Arial", 12), 
                                         fg_color="transparent", text_color="white", 
                                         hover_color="#4A8ABF", width=150, command=self.abrir_tela_cadastro)
        btn_cadastro.grid(row=5, column=0, columnspan=3, pady=(5, 10))
        
        # Status Firebase
        status_color = "#4CAF50" if self.firebase_ok else "#F44336"
        status_text = "✅ Conectado ao Banco de Dados" if self.firebase_ok else "❌ Banco de Dados Offline"
        
        ctk.CTkLabel(frame_form, text=status_text, fg_color="transparent", 
                      text_color=status_color, font=("Arial", 10)).grid(row=6, column=0, columnspan=3, pady=(0, 10))
        
        # Configurações de foco
        self.janela.after(100, lambda: self.entrada_usuario.focus())
        self.janela.bind('<Return>', lambda event: self.fazer_login())
    
    def abrir_tela_cadastro(self):
        TelaCadastro(self.janela)
    
    def fazer_login(self):
        usuario = self.entrada_usuario.get().strip()
        senha = self.entrada_senha.get()
        cargo_selecionado = self.cargo_selecionado.get()
        
        if not usuario or not senha:
            messagebox.showwarning("Erro de Login", "Preencha usuário e senha!")
            return
        
        if not self.firebase_ok:
            messagebox.showerror("Erro", "Banco de dados não conectado!")
            return
        
        try:
            usuarios = self.db.child('usuarios').get() 
            
            dados_usuario = next(
                (user_data for user_id, user_data in usuarios.items() 
                 if user_data.get('nome') == usuario or 
                    user_data.get('email') == usuario or 
                    user_data.get('usuario') == usuario),
                None
            )
            
            if dados_usuario:
                if dados_usuario.get('senha') == senha:
                    nivel_usuario_bd = dados_usuario.get('nivel_acesso', 'Funcionário')
                    
                    if nivel_usuario_bd.lower() == cargo_selecionado.lower() or nivel_usuario_bd == "Administrador":
                        
                        messagebox.showinfo("✅ Login Bem-sucedido", 
                                             f"Bem-vindo, {dados_usuario.get('nome', usuario)}!\nCargo: {nivel_usuario_bd}")
                        
                        # --- LÓGICA DE REDIRECIONAMENTO CORRIGIDA ---
                        self.janela.destroy() # Fecha a janela de login
                        
                        if nivel_usuario_bd == "Administrador":
                            self.abrir_sistema_principal(dados_usuario)
                        else: # Funcionário
                            self.abrir_sistema_funcionario(dados_usuario)
                        # --------------------------------------------
                        
                    else:
                        messagebox.showerror("Acesso Negado", 
                                             f"Este usuário é '{nivel_usuario_bd}', mas tentou logar como '{cargo_selecionado}'.")
                else:
                    messagebox.showerror("Erro de Login", "Senha incorreta!")
            else:
                messagebox.showerror("Erro de Login", "Usuário não encontrado!")
                
        except Exception as e:
            messagebox.showerror("Erro de Conexão", f"Falha ao consultar banco de dados:\n{str(e)}")
    
    def abrir_sistema_principal(self, dados_usuario):
        """Abre a tela do administrador."""
        tela_principal = TelaPrincipalAdmin(dados_usuario)
        tela_principal.iniciar()

    def abrir_sistema_funcionario(self, dados_usuario):
        """Novo método para abrir a tela do funcionário."""
        tela_funcionario = TelaSistemaFuncionario(dados_usuario)
        tela_funcionario.iniciar()
if __name__ == "__main__":
    # O script começa AQUI, chamando apenas a tela de Login
    app = LoginApp()

✅ Firebase Realtime Database conectado!
